# Rheumatoid Arthritis Synovial Fluid: Pre- vs Post-Treatment Analysis

**BioOptimize-scMultiomics Example Project 2**

This notebook analyzes the dynamic effects of TNF-α (adalimumab) and JAK (tofacitinib) inhibitors on synovial fluid immune cells in RA patients.

## Dataset
- **Accession**: GSE296117 (GEO)
- **Paper**: He et al., *Scientific Data* (2025) — A single-cell RNA-seq dataset of synovial fluid from rheumatoid arthritis treated with TNF-α/JAK inhibitor
- **Design**: Paired synovial fluid samples from RA patients before and after 1 month of treatment (adalimumab or tofacitinib + MTX)
- **Cells**: ~100,000 high-quality cells after QC
- **Link**: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE296117

### Data Availability
- Processed Seurat RDS object: `GSE296117_RA_geo.rds` (~2.3 GB) available as GEO supplementary file
- Raw data under controlled access (GSA-human HRA011646)

**Download processed RDS:**
```bash
# From GEO supplementary files
wget https://ftp.ncbi.nlm.nih.gov/geo/series/GSE296nnn/GSE296117/suppl/GSE296117_RA_geo.rds.gz
gunzip GSE296117_RA_geo.rds.gz
```

Place the file in `data/raw/GSE296117/`.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white')

print('Scanpy version:', sc.__version__)

## 1. Load Data

Option A: Convert Seurat RDS to AnnData (requires R/reticulate or pre-converted h5ad).
Option B: Use a pre-converted AnnData if available.

For this example we show the typical workflow assuming an AnnData object with treatment metadata.

In [ ]:
# Example loading path
data_path = Path('data/raw/GSE296117/RA_synovial_fluid.h5ad')

if data_path.exists():
    adata = sc.read_h5ad(data_path)
    print(f'Loaded: {adata.n_obs} cells, {adata.n_vars} genes')
    print('Metadata columns:', list(adata.obs.columns))
else:
    print('Data not found. Download and convert RDS first.')
    print('Expected metadata columns: treatment (pre/post), drug (adalimumab/tofacitinib), patient, etc.')
    # Demo structure for illustration
    adata = None

## 2. Quality Control & Preprocessing

In [ ]:
if adata is not None:
    # Standard QC
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
                 groupby='treatment', jitter=0.4)
    
    # Filter
    sc.pp.filter_cells(adata, min_genes=200)
    adata = adata[adata.obs.pct_counts_mt < 20, :].copy()
    
    print(f'After QC: {adata.n_obs} cells')

## 3. Integration & Clustering

Correct for patient/batch effects while preserving treatment signal.

In [ ]:
if adata is not None:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key='patient')
    adata = adata[:, adata.var.highly_variable].copy()
    
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=30)
    
    # Harmony integration on patient
    sc.external.pp.harmony_integrate(adata, key='patient', basis='X_pca')
    
    sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_neighbors=15)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=0.5)

## 4. Treatment Effect Visualization

In [ ]:
if adata is not None:
    # UMAP by treatment and drug
    sc.pl.umap(adata, color=['leiden', 'treatment', 'drug', 'patient'], ncols=2)
    
    # Key pathogenic markers in RA synovial fluid
    markers = ['SPP1', 'S100A12', 'CXCL13', 'CD4', 'CD14', 'CD68',
               'IL1B', 'TNF', 'STAT1', 'FOXP3']
    sc.pl.umap(adata, color=markers, ncols=3)
    
    # Rank genes
    sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
    sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

## 5. Pre- vs Post-Treatment Comparison

In [ ]:
if adata is not None:
    # Composition changes
    composition = pd.crosstab(adata.obs['leiden'], adata.obs['treatment'], normalize='columns')
    print('Cluster composition by treatment:')
    print(composition.round(3))
    
    # Differential expression between pre and post (within major clusters)
    # Example: focus on myeloid clusters
    # sc.tl.rank_genes_groups(adata, 'treatment', groups=['post'], reference='pre', method='wilcoxon')
    
    # Save
    adata.write('results/RA_SF_treatment_processed.h5ad')
    print('Analysis complete.')

## Summary & Biological Insights

This example demonstrates analysis of treatment response in RA synovial fluid:

**Key findings typically observed in this dataset:**
- Reduction in pathogenic SPP1+ macrophages and CXCL13+ CD4+ T cells after treatment
- Drug-specific effects (tofacitinib often shows stronger disruption of cell-cell communication)
- Changes in JAK/STAT and inflammatory pathways

**Workflow highlights:**
- Paired pre/post design
- Patient-level batch correction with Harmony
- Treatment effect visualization and composition analysis

**Extensions for a full project:**
- Cell type annotation with reference atlases
- Ligand-receptor analysis (LIANA / CellPhoneDB)
- Trajectory analysis of macrophage states
- Integration with clinical response (ACR20)

Research use only.